In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
f='/glade/derecho/scratch/juliob/NoFGW_latMM_fmt_c64144/run/NoFGW_latMM_fmt_c64144.cam.h1a.1985-06.nc'



f0='/glade/derecho/scratch/juliob/archive/CTL_fmt_c64144/atm/hist/CTL_fmt_c64144.cam.h0a.1990-01.nc'
X0=xr.open_dataset( f0 )

f2='/glade/derecho/scratch/juliob/archive/CTL_fmt_c64144/atm/hist/CTL_fmt_c64144.cam.h2i.1990-01-01-00000.nc'
X2=xr.open_dataset( f2 )







In [ ]:
omg_d = X2.OMEGA.values
u_d = X2.U.values
v_d = X2.V.values
pint_d = X2.PINT.values
lat = X2.lat.values
lon = X2.lon.values
omg_m = X0.OMEGA.values
lat = X0.lat.values
lon = X0.lon.values
lev = X0.lev.values




In [ ]:
X2

In [ ]:
print(lev[15])

In [ ]:
fig,axs = plt.subplots( 1, 2, figsize=(20,7) )
z=10
olv=np.linspace( -0.005, 0.005, num=51 )
ax=axs[0]
c=ax.tricontourf( lon,lat, omg_d[0,z,:],cmap='bwr' , levels=0.1*olv )
ax.set_title( 'Instantaneous Jan 1, 1990' )
plt.colorbar( c )
ax=axs[1]
c=ax.tricontourf( lon,lat, omg_m[0,z,:],cmap='bwr' , levels=0.1*olv  )
ax.set_title( 'Jan 1990 mean' )
plt.colorbar( c )


In [ ]:

RgObs={}
Dst = 'fv1x1'
Src = 'ne30pg3'
DstInfo = GrU.gridInfo(Dst) #,Vgrid=DstVgrid)
lat_a,lon_a = GrU.latlon( scrip= DstInfo['scrip'], Hkey=DstInfo['Hkey'] )
RegridObj = GrU.regrid_object_lib(RgOb=RgObs, src=Src, dst=Dst,RegridMethod='bilinear')



#AAa_s_xy=RgF.Horz(xfld_Src=AAa_s , Src=Src, Dst=Dst , RegridObj_In= RegridObj )


In [ ]:
omg_d_yx=RgF.Horz(xfld_Src=omg_d , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
omg_m_yx=RgF.Horz(xfld_Src=omg_m , Src=Src, Dst=Dst , RegridObj_In= RegridObj )


u_d_yx=RgF.Horz(xfld_Src=u_d , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
v_d_yx=RgF.Horz(xfld_Src=v_d , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
pint_d_yx=RgF.Horz(xfld_Src=pint_d , Src=Src, Dst=Dst , RegridObj_In= RegridObj )


In [ ]:
fig,axs = plt.subplots( 1, 2, figsize=(20,7) )
z=10
olv=np.linspace( -0.005, 0.005, num=51 )
ax=axs[0]
c=ax.contourf( lon_a,lat_a, omg_d_yx[0,z,:],cmap='bwr' , levels=0.1*olv )
ax.set_title( 'Instantaneous Jan 1, 1990' )
plt.colorbar( c )
ax=axs[1]
c=ax.contourf( lon_a,lat_a, omg_m_yx[0,z,:],cmap='bwr' , levels=0.1*olv  )
ax.set_title( 'Jan 1990 mean' )
plt.colorbar( c )


In [ ]:
zlev=-7_000.*np.log( lev/1_000.)



fig,axs = plt.subplots( 1, 2, figsize=(20,7) )
z=10
olv=0.1*np.linspace( -0.005, 0.005, num=51 )
ax=axs[0]
c=ax.contourf( lat_a, zlev, np.mean(omg_d_yx[0,:,:,:],axis=2) ,cmap='bwr' , levels=0.1*olv )
ax.set_title( 'Instantaneous Jan 1, 1990' )
plt.colorbar( c )
ax=axs[1]
c=ax.contourf( lat_a, zlev, np.mean(omg_m_yx[0,:,:,:],axis=2),cmap='bwr' , levels=0.1*olv  )
ax.set_title( 'Jan 1990 mean' )
plt.colorbar( c )


In [ ]:
nt,nz,ny,nx = np.shape(u_d_yx)

In [ ]:
delp_d_yx=pint_d_yx[:,1:,:,:]- pint_d_yx[:,0:-1,:,:]
print(np.shape(pint_d_yx))
print(np.shape(delp_d_yx))
#print(np.shape(pint_d_yx[:,0:-1,:,:]    ))
omi_d_yx=np.zeros( (nt,nz+1,ny,nx) )
div_d_yx=np.zeros( (nt,nz,ny,nx) )


In [ ]:
plt.plot(delp_d_yx[0,:,100,100] )

In [ ]:
importlib.reload(nuti)

for t in np.arange(nt): 
    for z in np.arange( nz ):
        div=nuti.Sphere_Div2( f_x=u_d_yx[t,z,:,:], f_y=v_d_yx[t,z,:,:], lat=lat_a, lon=lon_a, wrap=True )
        div_d_yx[t,z,:,:] = div

In [ ]:
for t in np.arange(nt): 
    for z in np.arange( start=1,stop=nz+1 ):
         omi_d_yx[t,z,:,:] = omi_d_yx[t,z-1,:,:] - div_d_yx[t,z-1,:,:] * delp_d_yx[t,z-1,:,:]


In [ ]:
t,z=0,0
#plt.contour(   div_d_yx[t,z,:,:] * delp_d_yx[t,z,:,:]   )
plt.plot(   omi_d_yx[t,z+10,:,:] .flatten()   )
#plt.plot(   (div_d_yx[t,z,:,:] * delp_d_yx[t,z,:,:]).flatten()   )


In [ ]:
fig,axs = plt.subplots( 1, 2, figsize=(20,7) )
z=10
print(lev[10])
olv=np.linspace( -0.005, 0.005, num=51 )
ax=axs[0]
#c=ax.contourf( lon_a,lat_a, omg_d_yx[0,z,:],cmap='bwr' , levels=0.1*olv )
c=ax.contourf( lon_a,lat_a, omg_d_yx[0,z,:,:],cmap='bwr' , levels=0.1*olv )
ax.set_title( f'Instantaneous {r"$\omega$"} at {lev[10]:.2f} hPa Jan 1, 1990' )
plt.colorbar( c )
ax=axs[1]
#c=ax.contourf( lon_a,lat_a, div_d_yx[0,z,:],cmap='bwr' , levels=21  )
c=ax.contourf( lon_a,lat_a, 0.5*(omi_d_yx[0,z,:,:]+omi_d_yx[0,z+1,:,:]),cmap='bwr' , levels=0.1*olv  )
ax.set_title( f'Offline calculation of {r"$\omega$"} using lat-lon regridded SE output \n with {r"$\int \nabla \cdot \vec{u}~ dp$"}' )
plt.colorbar( c )


In [ ]:
print( poopypants )

In [ ]:
f='/glade/derecho/scratch/juliob/archive/f.e13.FAMIPC5.ne120_ne120_mt12.DECK_1950TR_RCP85_CDR.006/atm/regridded/f.e13.FAMIPC5.ne120_ne120_mt12.DECK_1950TR_RCP85_CDR.006.cam.h0.2006-03.nc'
f='/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_134.FHISTC_LTso.ne30_L58.001/atm/hist/f.e30_cam6_4_134.FHISTC_LTso.ne30_L58.001.cam.h0a.1999-08.nc'
X=xr.open_dataset( f )


In [ ]:
print( X.lon.values[-1] )

In [ ]:
Ftopo178 = '/glade/campaign/cgd/amp/pel/topo/cesm3/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0050_noleak_20250325.nc'

Ftopo156 = '/glade/campaign/cgd/amp/pel/topo/cesm3/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0050_noleak_20250409_28kmGRISrough.nc'




Topo156 = xr.open_dataset( Ftopo156 )
Topo178 = xr.open_dataset( Ftopo178 )


In [ ]:

sgh30_156=Topo156.SGH30.values
sgh30_178=Topo178.SGH30.values
lat=Topo156.lat.values
lon=Topo156.lon.values


In [ ]:
fig,axs=plt.subplots( 1,3, figsize=(28,5) )
levs=np.linspace(0,1000,num=51 )
dlevs=np.linspace(-200,200,num=51 )

ax=axs[0]
co=ax.tricontourf( lon, lat, sgh30_156, levels=levs ,cmap='terrain' )
plt.colorbar( co )
ax=axs[1]
co=ax.tricontourf( lon, lat, sgh30_178, levels=levs, cmap='terrain' )
plt.colorbar( co )
ax=axs[2]
co=ax.tricontourf( lon, lat, sgh30_178-sgh30_156, levels=dlevs, cmap='bwr'  )
plt.colorbar( co )


In [ ]:
f='/glade/derecho/scratch/juliob/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/run/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001.cpl.r.2001-10-01-00000.nc'
Y=xr.open_dataset( f )

In [ ]:
print(Y.ocnExpAccum_cnt.values)

In [ ]:
X=xr.open_dataset( '/glade/derecho/scratch/juliob/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/run/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001.docn.r.2001-07-01-00000.nc')
X2=  xr.open_dataset('/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/rest/2000-07-01-00000/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001.docn.r.2000-07-01-00000.nc' )


In [ ]:
X

In [ ]:
X.coords

In [ ]:
for koo in X.coords:
    print(koo, X[koo].values==X2[koo].values    )

In [ ]:
for koo in X:
    print(koo, X[koo].values==X2[koo].values    )

In [ ]:
f='/glade/work/juliob/Topo/NCARTopoMaster/cases/ne120pg3_Sco25_GrnlAntSGH30x/output/ne120pg3_gmted2010_modis_bedmachine_nc3000_Laplace0025_noleak_greenlndantarcsgh30fac2.50_20251009.nc'
#f='/glade/derecho/scratch/juliob/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001.cam.h2a.1996-08-30-00000.nc'
X=xr.open_dataset(f)

In [ ]:

lat=X.lat.values
lon=X.lon.values
topo=X.PHIS.values


In [ ]:
plt.tricontourf( lon, lat, topo/9.8, levels=[0,1,10,100,200,500,1000], cmap='terrain')
plt.colorbar()
plt.xlim(280,320)
plt.ylim(5,30)

In [ ]:
ts=X.TS.values
time=X.time.values

In [ ]:
fig,axs=plt.subplots( 1, 2, figsize=(20,8))
levs=np.linspace(170,330,num=81)
ax=axs[0]
c=ax.contourf( ts[0,:,:] ,levels=levs, cmap='gist_ncar')
plt.colorbar(c)
ax=axs[1]
c=ax.contourf( ts[9,:,:] ,levels=levs, cmap='gist_ncar')
plt.colorbar(c)



In [ ]:
plt.plot(ts[:,:,:].flatten() )


In [ ]:
print(time)

In [ ]:
#f='/glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/USGS-gtopo30_ne120np4_16xdel2-PFC-consistentSGH.nc'
f='/glade/campaign/cgd/amp/pel/topo/cesm3/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0050_noleak_20250325.nc' # exp 188
f='/glade/campaign/cgd/amp/pel/topo/cesm3/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0100_noleak_greendlndantarcsgh30fac2.50_20250708.nc' # exp 191
f='/glade/work/juliob/Topo/NCARTopoMaster/cases/ne120pg3_Sco25_GrnlAntSGH30x/output/ne120pg3_gmted2010_modis_bedmachine_nc3000_Laplace0025_noleak_greenlndantarcsgh30fac2.50_20251009.nc'
SSTICE_DATA_FILENAME='/glade/work/xueliu/data/cesm_input/SST/SSTinput_CDR_noleap/sst_input_0.25_CDR-19900101-19901231.nc'
f0='/glade/work/xueliu/data/cesm_input/SST/SSTinput_CDR_noleap/sst_input_0.25_CDR-19900101-19901231.nc'

#f2='/glade/work/juliob/Topo/NCARTopoJTB/cases/ne120pg3_Sco25_test/output/ne120pg3_gmted2010_modis_bedmachine_nc3000_Laplace0025_noleak_greenlndantarcsgh30fac2.50_20251009.nc'

f2 = '/glade/derecho/scratch/juliob/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.JTB.001/run/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.JTB.001.cam.h2a.1990-01-02-00000.nc'

X1 = xr.open_dataset('/glade/derecho/scratch/juliob/SSTinput_CDR_noleap/decades_cdf5/sst_input_0.25_CDR-19890101-20011231.nc')
X0=xr.open_dataset( f0 )
X2=xr.open_dataset( f2 )


In [ ]:
X2

In [ ]:
lat0=X0.lat.values
lon0=X0.lon.values
time0=X0.time.values
sst0=X0.SST_cpl.values
ice0=X0.ice_cov.values
ice0=X0.ice_cov.values

lat1=X1.lat.values
lon1=X1.lon.values
time1=X1.time.values
sst1=X1.SST_cpl.values
ice0=X0.ice_cov.values

lat2=X2.lat.values
lon2=X2.lon.values
time2=X2.time.values
sst2=X2.SST.values
ts2=X2.TS.values
ice2=X2..values




In [ ]:
t=15
#t=1
print( time0[t] )
print( time1[t+365] )
print( time2[t] )

In [ ]:
fig,axs=plt.subplots( 1,3, figsize=(37,9) )
levs=np.linspace(270,310,num=41) - 273.
lonlim=[100,300]
latlim=[0,90]

ax=axs[0]
c=ax.contourf( lon0, lat0, sst0[t,:,:], levels= levs, extend='both' ,cmap='terrain')
ax.set_xlim(lonlim )
ax.set_ylim(latlim )
ax.set_title("Original SSTs from Xueliu Jan 16 1990 (deg C) ")

plt.colorbar(c)

ax=axs[1]
c=ax.contourf( lon1, lat1, sst1[365+t,:,:], levels= levs, extend='both' ,cmap='terrain')
ax.set_xlim(lonlim )
ax.set_ylim(latlim ) 
ax.set_title("Reprocessed decadal SSTs Jan 16 1990 (deg C) ")
plt.colorbar(c)

ax=axs[2]
c=ax.contourf( lon2, lat2, sst2[t,:,:]-273.15, levels= levs, extend='both' ,cmap='terrain')
ax.set_xlim(lonlim )
ax.set_ylim(latlim )
ax.set_title("SSTs in CAM run history (cam.h2a) Jan 16 1990 (deg C)")
plt.colorbar(c)




In [ ]:

lat=X.lat.values
lon=X.lon.values
sgh=X.SGH.values
mxdis=X.MXDIS.values
mxdis2=X2.MXDIS.values



In [ ]:
plt.plot( mxdis[6,:] - 0*mxdis2[6,:] )

In [ ]:
fig,axs=plt.subplots( 1,3, figsize=(37,9) )

ax=axs[0]
c=ax.tricontourf( lon, lat, sgh, levels=np.linspace(1,1001,num=11), extend='both' ,cmap='terrain')
plt.colorbar(c)

ax=axs[1]
c=ax.tricontourf( lon, lat, mxdis[0,:], levels=np.linspace(1,2001,num=11), extend='both' ,cmap='terrain')
plt.colorbar(c)

ax=axs[2]
c=ax.tricontourf( lon, lat, mxdis[0,:]-2*sgh, levels=np.linspace(-1000,1000,num=10), extend='both' ,cmap='bwr')
plt.colorbar(c)




In [ ]:
plt.tricontourf( lon, lat, sgh, levels=np.linspace(1,1001,num=11), extend='both' ,cmap='terrain')
plt.colorbar()

In [ ]:
print( 228/12.)

In [ ]:
qrs_tot=X.QRS_TOT.values
lat=X.lat.values
zlev = -7.0*np.log(X.lev.values/1_000.)


In [ ]:
qrs_tot=qrs_tot.reshape( 228//12,12,70,192)

In [ ]:
qrs_tot_cyc = np.mean( qrs_tot, axis=0 )
qrs_tot_av = np.mean( qrs_tot_cyc, axis=0 )


In [ ]:



fig,axs=plt.subplots( 1,3, figsize=(16,5))
ax=axs[0]
co=ax.contourf(lat,zlev, 86_400.*qrs_tot_av[:,:], levels=1*np.linspace(0,10,num=11) )
ax.set_ylim(0,95)
cbar=plt.colorbar(co)
ax=axs[1]
co=ax.contourf(lat,zlev, 86_400.*qrs_tot_cyc[2,:,:], levels=1*np.linspace(0,10,num=11) )
ax.set_ylim(0,95)
ax=axs[2]
co=ax.contourf(lat,zlev, 86_400.*qrs_tot_cyc[6,:,:], levels=1*np.linspace(0,10,num=11) )
ax.set_ylim(0,95)
#plt.colorbar()
#plt.plot( X.QRS_TOT[0,:,7] * 86_400. ,-7*np.log(X.lev/1_000.) )




In [ ]:
plt.plot( 86_400.*qrs_tot_cyc[6,55,:] )

In [ ]:
plt.plot( 86_400.*qrs_tot_av[:,95] ,zlev )
plt.ylim(0,100)
plt.xlim(0,20)
plt.ylabel('Z')

In [ ]:
print(zlev[10])

In [ ]:
def f_LTE(p_hPa, p0=1.0, k=1.6):
    """
    Fraction of short-wave heating that behaves as LTE, as a function of pressure (hPa).
    p0 ≈ transition pressure (~1 hPa ≈ 50 km).
    k  controls sharpness (1.3–2.0 reasonable).
    Returns values in [0,1].
    """
    p = np.maximum(p_hPa, 1e-6)
    return 1.0 / (1.0 + (p0 / p)**k)


flte = f_LTE( X.lev[10:] ,p0=0.5 )

In [ ]:
#import numpy as np
from scipy.optimize import curve_fit

def model0(z, c0, c1, a1, z1, s1, a2, z2, s2):
    return c0 + c1*z + a1*np.tanh((z - z1)/s1) + a2*np.tanh((z - z2)/s2)

def model1(z, c0, c1, a1, z1, s1):
    return c0 + c1*z + a1*np.sin((z - z1)/s1)

p0 = [0.3, 0.07, 2.0, 15.0, 6.0, 1.8, 75.0, 8.0]  # start guesses
p1 = [0.3, 0.07, 1.0, 45.0, 6.0 ]  # start guesses

x0=86_400.*qrs_tot_av[10:,95]
z0=zlev[10:]
# z, x = your data arrays used to make the plot
popt, pcov = curve_fit(model, z0, x0, p0=p0, maxfev=20000)


x=(1.0-flte)*x0
x=x[0:21]
z=z0[0:21]
popt1, pcov1 = curve_fit(model1, z, x, p0=p1, maxfev=20000)

# popt holds best-fit parameters


In [ ]:
c0_0, c1_0, a1_0, z1_0, s1_0, a2_0, z2_0, s2_0 = popt
c0_1, c1_1, a1_1, z1_1, s1_1 = popt1
print( c0_1, c1_1, a1_1, z1_1, s1_1 )

In [ ]:
plt.plot( x0,z0 , label='mean WACCM QRS Forcing' )
plt.plot( model0(z0, c0_0, c1_0, a1_0, z1_0, s1_0, a2_0, z2_0, s2_0) , z0, label='functional fit 1' )
plt.plot( model1(z, c0_1, c1_1, a1_1, z1_1, s1_1 ) , z, '-o', label='functional fit 2' )
plt.plot( model1(z0, c0_1, c1_1, a1_1, z1_1, s1_1 ) , z0, label='functional fit 2, complete range' )
plt.xlabel('Q (K d$^{-1}$)' )
plt.ylabel('Z (km)' )
plt.legend()


In [ ]:
print( popt )

In [ ]:
x0=86_400.*qrs_tot_av[10:,95]
z=zlev[10:]
#nlte_frac=np.exp( z / 70. )


In [ ]:

plt.plot( (1.0-flte)*x0 , z)
plt.plot( x0 , z)


In [ ]:
#plt.contourf(X.QRS_TOT[0,:,:] )
plt.plot( X.QRS_TOT[0,:,7] * 86_400. ) #,-7*np.log(X.lev/1_000.) )


In [ ]:
plt.plot( X.QRS_TOT[:,15,7] * 86_400. )

In [ ]:
plt.tricontour(X.lon,X.lat,X.LANDFRAC,levels=[0.5,10.])
plt.xlim( 250,320 )
plt.ylim( 10,45)

In [ ]:
f='/glade/campaign/cesm/cesmdata/inputdata/atm/cam/chem/ubc/b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.ensAvg123.cam.h0zm.H2O.185001-201412_c230509.nc'

dir='/glade/derecho/scratch/juliob/radtest_Q02_mpas120_fmt_c6477/run/'
f=dir+'radtest_Q02_mpas120_fmt_c6477.cam.h1i.1985-01-01-01800.nc'

dir='/glade/derecho/scratch/juliob/radtest_AICN01hf_mpas120_fmt_c6477/run/'
f=dir+'radtest_AICN01hf_mpas120_fmt_c6477.cam.h1i.1985-01-05-46800.nc'
X=xr.open_dataset( f )

In [ ]:
X


In [ ]:
#te=X.T.values
lon=X.lon.values
lat=X.lat.values
u=X.U.values
utg=X.Target_U.values




In [ ]:
print(np.shape(u))
ulev=np.linspace( -200,200,num=51)

In [ ]:
plt.tricontourf( lon, lat, u[0,0,:], cmap='bwr_r' ,levels=ulev)
plt.colorbar()

In [ ]:
plt.plot(u[0,:,:].flatten())
#plt.plot(te3[0,0,:])


In [ ]:
plt.plot(te[0,0,:])
plt.plot(te3[0,0,:])
